<div align="center">

<img src="https://s3.amazonaws.com/files.pucp.edu.pe/pucp-general/img-header/logo-pucp-new.svg" alt="Pontificia Universidad Católica del Perú">

<br><br>

## Pontificia Universidad Católica del Perú  

### Deep Learning con Python  

<br>

## Semana 4 — Tarea 4

### Clasificacion de texto con RNN - LSTM vs GRU

</div>

---

## 1. Introducción

En esta tarea se aborda el problema de **clasificación de texto multicategoría** utilizando modelos de redes neuronales recurrentes (RNN). En particular, se emplea el dataset **AG News**, el cual contiene noticias clasificadas en cuatro categorías:

- 🌍 World  
- 🏅 Sports  
- 💼 Business  
- 🤖 Sci/Tech  

El objetivo principal es **comparar el rendimiento de dos arquitecturas avanzadas de RNN**:

- **LSTM (Long Short-Term Memory)**
- **GRU (Gated Recurrent Unit)**

Ambos modelos están diseñados para capturar dependencias en secuencias de texto, pero presentan diferencias en complejidad y eficiencia computacional.

---

## 2. Objetivos

### Objetivo general
Comparar el desempeño de modelos LSTM y GRU en la tarea de clasificación de texto.

### Objetivos específicos
- Implementar un pipeline completo de procesamiento de texto  
- Entrenar modelos basados en LSTM y GRU  
- Evaluar el rendimiento mediante métricas como accuracy, precision, recall y F1-score  
- Analizar las diferencias en desempeño y eficiencia entre ambos modelos  

---

## 3. Dataset

Se utiliza el dataset **AG News**, ampliamente empleado en tareas de clasificación de texto.

Características principales:
- Aproximadamente 120,000 ejemplos de entrenamiento  
- 7,600 ejemplos de prueba  
- Cuatro clases balanceadas  
- Cada instancia contiene un título y una descripción  

Para el preprocesamiento:
- Se concatenó el título y la descripción  
- Se realizó limpieza básica del texto  
- Se aplicó tokenización y padding  

---


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM, GRU, Bidirectional, Dropout
from sklearn.model_selection import train_test_split

## 📂 Carga de Datos

En esta sección se realiza la carga de los datasets necesarios para el desarrollo del modelo de clasificación de texto.

Se utilizan dos archivos en formato CSV:

- `train.csv`: contiene los datos de entrenamiento  
- `test.csv`: contiene los datos de prueba  

Ambos datasets corresponden al conjunto **AG News**, el cual incluye noticias etiquetadas en cuatro categorías.

Para su manipulación se emplea la librería **pandas**, permitiendo leer los archivos y almacenarlos en estructuras tipo DataFrame para su posterior procesamiento.

Finalmente, se visualizan las primeras filas del dataset de entrenamiento para verificar su correcta carga y estructura.

In [ ]:
train_df = None
test_df = None

## 🧠 Actividad: Preparación del Texto y Etiquetas

En esta sección deberás preparar los datos para su uso en el modelo de clasificación.

---

### 🔹 1. Construcción del texto de entrada

El dataset contiene dos columnas importantes:

- `Title`
- `Description`

👉 **Instrucción:**  
Crea una nueva columna llamada `text` que combine ambas columnas (título + descripción), separadas por un espacio.

Esto permitirá que el modelo tenga mayor contexto al analizar cada noticia.

---

### 🔹 2. Preparación de las etiquetas

La columna `Class Index` contiene las clases, pero sus valores van de **1 a 4**.

👉 **Instrucción:**  
Convierte estas etiquetas para que comiencen en **0**, restando 1 a cada valor.

Debes crear:

- `y_train` a partir de `train_df`
- `y_test` a partir de `test_df`

---

### ⚠️ Consideraciones

- Este paso es necesario para que el modelo funcione correctamente en Keras  
- Asegúrate de no modificar los DataFrames originales incorrectamente  
- Verifica que las etiquetas finales estén en el rango **0 a 3**

---

✅ Al finalizar esta sección, debes tener:
- Una columna `text` en ambos datasets  
- Variables `y_train` y `y_test` correctamente definidas  

In [ ]:
y_train = None
y_test = None

## 🔤 Actividad: Tokenización y Padding

En esta sección deberás transformar el texto en una representación numérica para poder entrenar el modelo.

---

### 🔹 1. Definir parámetros

👉 **Instrucción:**  
Define las siguientes variables:

- `vocab_size`: tamaño máximo del vocabulario (usa 20000)
- `max_len`: longitud máxima de las secuencias (usa 100)

---

### 🔹 2. Crear el Tokenizer

👉 **Instrucción:**  
Crea un `Tokenizer` que:

- Limite el vocabulario usando `vocab_size`
- Incluya un token especial para palabras desconocidas (`"<OOV>"`)

Luego, ajusta (`fit`) el tokenizer utilizando únicamente los textos de entrenamiento.

---

### 🔹 3. Convertir texto a secuencias

👉 **Instrucción:**  
Transforma los textos en secuencias de números:

- `X_train` a partir de `train_df["text"]`
- `X_test` a partir de `test_df["text"]`

---

### 🔹 4. Aplicar padding

👉 **Instrucción:**  
Asegura que todas las secuencias tengan la misma longitud (`max_len`) utilizando padding al final (`padding="post"`).

---

### ⚠️ Consideraciones importantes

- No debes ajustar el tokenizer con el conjunto de prueba  
- El padding es necesario para que el modelo procese las secuencias correctamente  
- Las secuencias más largas serán truncadas y las más cortas serán rellenadas  

---

✅ Al finalizar esta sección, debes tener:

- `X_train` y `X_test` como matrices numéricas  
- Todas las secuencias con longitud fija (`max_len`)  

In [ ]:
x_train = None
x_test = None

## 🧠 Actividad: Carga de Embeddings Preentrenados (GloVe)

En esta sección deberás cargar vectores de palabras preentrenados utilizando **GloVe (Global Vectors for Word Representation)**.

Estos embeddings permiten representar cada palabra como un vector numérico, capturando relaciones semánticas entre palabras.

---

### 🔹 1. Definir la dimensión del embedding

👉 **Instrucción:**  
Define la variable `embedding_dim` con valor **100**, ya que se utilizará el archivo `glove.6B.100d.txt`.

---

### 🔹 2. Crear estructura de almacenamiento

👉 **Instrucción:**  
Crea un diccionario vacío llamado `embedding_index` donde se almacenarán los vectores de cada palabra.

---

### 🔹 3. Leer el archivo GloVe

👉 **Instrucción:**  
Abre el archivo `glove.6B.100d.txt` y recorre cada línea para:

- Separar la palabra del vector  
- Convertir los valores numéricos a tipo `float32`  
- Almacenar cada palabra junto a su vector en el diccionario `embedding_index`  

---

### ⚠️ Consideraciones importantes

- Asegúrate de que el archivo `glove.6B.100d.txt` esté en el mismo directorio del notebook  
- Este proceso puede tardar unos segundos debido al tamaño del archivo  
- Cada palabra quedará asociada a un vector de dimensión 100  

---

### 🎯 Objetivo

Al finalizar esta sección, debes tener un diccionario (`embedding_index`) que mapea palabras a sus representaciones vectoriales.

Esto será utilizado posteriormente para construir la matriz de embeddings del modelo.

## 🧩 Actividad: Construcción de la Matriz de Embeddings

En esta sección deberás construir la **matriz de embeddings**, la cual permitirá integrar los vectores preentrenados de GloVe dentro del modelo.

---

### 🔹 1. Obtener el índice de palabras

👉 **Instrucción:**  
Obtén el diccionario `word_index` a partir del tokenizer previamente entrenado.

Este diccionario asigna un índice numérico a cada palabra del vocabulario.

---

### 🔹 2. Inicializar la matriz de embeddings

👉 **Instrucción:**  
Crea una matriz de ceros llamada `embedding_matrix` con dimensiones:

- Número de filas: `vocab_size`  
- Número de columnas: `embedding_dim`  

Cada fila representará el vector de una palabra.

---

### 🔹 3. Asignar vectores de GloVe

👉 **Instrucción:**  
Recorre el diccionario `word_index` y, para cada palabra:

- Verifica que su índice sea menor que `vocab_size`  
- Busca su vector en `embedding_index`  
- Si el vector existe, asígnalo en la posición correspondiente de `embedding_matrix`  

---

### ⚠️ Consideraciones importantes

- No todas las palabras del dataset estarán en GloVe  
- Las palabras no encontradas permanecerán como vectores de ceros  
- Este proceso alinea el vocabulario del tokenizer con los embeddings preentrenados  

---

### 🎯 Objetivo

Al finalizar esta sección, debes tener una matriz (`embedding_matrix`) lista para ser utilizada en la capa **Embedding** del modelo.

Esta matriz permitirá inicializar el modelo con conocimiento semántico previo.

## 🧠 Actividad: Construcción del Modelo LSTM

En esta sección deberás implementar un modelo basado en **LSTM (Long Short-Term Memory)** para la clasificación de texto.

---

### 🔹 1. Definir la arquitectura del modelo

👉 **Instrucción:**  
Construye un modelo secuencial (`Sequential`) con las siguientes capas:

---

### 🔸 Capa 1: Embedding

- Tamaño del vocabulario: `vocab_size`  
- Dimensión de los embeddings: `embedding_dim`  
- Pesos iniciales: `embedding_matrix` (GloVe)  
- Longitud de entrada: `max_len`  
- `trainable=False` (no actualizar embeddings durante el entrenamiento)  

👉 Esta capa convierte las palabras en vectores numéricos utilizando embeddings preentrenados.

---

### 🔸 Capa 2: LSTM

- Unidades: **48**

👉 Esta capa permite capturar dependencias secuenciales en el texto.

---

### 🔸 Capa 3: Dropout

- Tasa: **0.3**

👉 Se utiliza para reducir el overfitting apagando aleatoriamente neuronas durante el entrenamiento.

---

### 🔸 Capa 4: Dense (oculta)

- Unidades: **32**  
- Activación: **ReLU**

👉 Permite aprender combinaciones no lineales de las características extraídas.

---

### 🔸 Capa 5: Dense (salida)

- Unidades: **4** (una por cada clase)  
- Activación: **Softmax**

👉 Genera la probabilidad de pertenencia a cada clase.

---

### 🔹 2. Compilación del modelo

👉 **Instrucción:**  
Compila el modelo utilizando:

- Función de pérdida: `sparse_categorical_crossentropy`  
- Optimizador: `adam`  
- Métrica: `accuracy`  

---

### 🔹 3. Resumen del modelo

👉 **Instrucción:**  
Muestra el resumen del modelo para verificar su estructura y número de parámetros.

---

### ⚠️ Consideraciones importantes

- Se utiliza `sparse_categorical_crossentropy` porque las etiquetas están codificadas como enteros (no one-hot)  
- La capa Embedding no es entrenable (`trainable=False`) para aprovechar los embeddings de GloVe  
- El número de unidades y capas puede ajustarse en experimentos posteriores  

---

### 🎯 Objetivo

Implementar un modelo LSTM funcional para clasificación de texto que sirva como base para la comparación con otros modelos.

In [ ]:
model_lstm = None

## 🧠 Actividad: Construcción del Modelo GRU

En esta sección deberás implementar un modelo basado en **GRU (Gated Recurrent Unit)** para la clasificación de texto.

---

### 🔹 1. Definir la arquitectura del modelo

👉 **Instrucción:**  
Construye un modelo secuencial (`Sequential`) con las siguientes capas:

---

### 🔸 Capa 1: Embedding

- Tamaño del vocabulario: `vocab_size`  
- Dimensión de los embeddings: `embedding_dim`  
- Pesos iniciales: `embedding_matrix` (GloVe)  
- Longitud de entrada: `max_len`  
- `trainable=False`  

👉 Esta capa transforma las palabras en vectores utilizando embeddings preentrenados.

---

### 🔸 Capa 2: GRU

- Unidades: **48**

👉 Esta capa es una variante de LSTM más eficiente, diseñada para capturar dependencias en secuencias con menor costo computacional.

---

### 🔸 Capa 3: Dropout

- Tasa: **0.3**

👉 Reduce el sobreajuste durante el entrenamiento.

---

### 🔸 Capa 4: Dense (oculta)

- Unidades: **32**  
- Activación: **ReLU**

👉 Permite aprender representaciones más complejas.

---

### 🔸 Capa 5: Dense (salida)

- Unidades: **4**  
- Activación: **Softmax**

👉 Genera probabilidades para cada clase.

---

### 🔹 2. Compilación del modelo

👉 **Instrucción:**  
Compila el modelo utilizando:

- Función de pérdida: `sparse_categorical_crossentropy`  
- Optimizador: `adam`  
- Métrica: `accuracy`  

---

### 🔹 3. Resumen del modelo

👉 **Instrucción:**  
Muestra el resumen del modelo para verificar su arquitectura.

---

### ⚠️ Consideraciones importantes

- GRU es más simple que LSTM, por lo que suele entrenar más rápido  
- Mantiene un rendimiento competitivo en tareas de NLP  
- Se utiliza la misma arquitectura base para permitir una comparación justa con el modelo LSTM  

---

### 🎯 Objetivo

Implementar un modelo GRU y compararlo posteriormente con el modelo LSTM en términos de desempeño y eficiencia.

In [ ]:
model_gru = None

## 🚀 Actividad: Entrenamiento de los Modelos

En esta sección deberás entrenar los modelos **LSTM** y **GRU** utilizando los datos preparados previamente.

---

### 🔹 1. Definir hiperparámetros

👉 **Instrucción:**  
Define los siguientes hiperparámetros de entrenamiento:

- `epochs`: número de épocas (**usar 20**)  
- `batch_size`: tamaño del lote (**usar 128**)  

---

### 🔹 2. Entrenamiento del modelo LSTM

👉 **Instrucción:**  
Entrena el modelo `model_lstm` utilizando:

- Datos de entrada: `X_train`  
- Etiquetas: `y_train`  
- Validación: `validation_split=0.2` (20% de los datos para validación)  
- Número de épocas: `epochs`  
- Tamaño de batch: `batch_size`  
- Mostrar progreso: `verbose=1`  

Guarda el historial de entrenamiento en una variable llamada `history_lstm`.

---

### 🔹 3. Entrenamiento del modelo GRU

👉 **Instrucción:**  
Entrena el modelo `model_gru` con los mismos parámetros utilizados en el modelo LSTM.

Guarda el historial de entrenamiento en una variable llamada `history_gru`.

---

### ⚠️ Consideraciones importantes

- Ambos modelos deben entrenarse con **los mismos hiperparámetros** para asegurar una comparación justa  
- El uso de `validation_split` permite evaluar el modelo durante el entrenamiento  
- El historial (`history`) será utilizado posteriormente para graficar métricas  

---

### 🎯 Objetivo

Entrenar ambos modelos (LSTM y GRU) y obtener sus historiales de entrenamiento para analizar su desempeño en las siguientes secciones.

In [ ]:
history_lstm = None
history_gru = None

## 📊 Actividad: Evaluación y Visualización de Resultados

En esta sección deberás evaluar el rendimiento de los modelos entrenados y visualizar su comportamiento durante el entrenamiento.

---

### 🔹 1. Evaluación en el conjunto de prueba

👉 **Instrucción:**  
Evalúa ambos modelos (`LSTM` y `GRU`) utilizando el conjunto de prueba (`X_test`, `y_test`).

Debes mostrar los resultados de:
- Loss  
- Accuracy  

---

### 🔹 2. Visualización de métricas

👉 **Instrucción:**  
Utiliza los historiales de entrenamiento (`history_lstm` y `history_gru`) para graficar:

#### 📈 Accuracy
- Accuracy de entrenamiento  
- Accuracy de validación  

#### 📉 Loss
- Loss de entrenamiento  
- Loss de validación  

---

### 🔹 3. Requisitos de las gráficas

- Debes generar **dos gráficas por modelo**:
  - Accuracy vs Epochs  
  - Loss vs Epochs  

- Cada gráfica debe incluir:
  - Curva de entrenamiento  
  - Curva de validación  
  - Título descriptivo  
  - Etiquetas en los ejes  
  - Leyenda  

---

### ⚠️ Consideraciones importantes

- Las gráficas deben permitir comparar el comportamiento del modelo durante el entrenamiento  
- Observa si existe **overfitting** (cuando la validación empeora mientras el entrenamiento mejora)  
- Analiza cuál modelo presenta mejor estabilidad y rendimiento  

---

### 🎯 Objetivo

Evaluar el desempeño final de los modelos y analizar visualmente su proceso de aprendizaje mediante curvas de accuracy y loss.

## 🔮 Actividad: Predicción con Nuevos Textos

En esta sección deberás utilizar los modelos entrenados para realizar predicciones sobre nuevos ejemplos de texto.

---

### 🔹 1. Construcción de la función de predicción

👉 **Instrucción:**  
Implementa una función llamada `predict_text` que reciba:

- Un modelo entrenado (`model`)
- Un texto de entrada (`text`)

La función debe realizar los siguientes pasos:

1. Convertir el texto en secuencia numérica usando el `tokenizer`  
2. Aplicar padding con la misma longitud (`max_len`) utilizada en el entrenamiento  
3. Realizar la predicción con el modelo  
4. Obtener la clase predicha utilizando `argmax`  
5. (Opcional) Mostrar las probabilidades de cada clase  

---

### 🔹 2. Realizar predicciones

👉 **Instrucción:**  
Utiliza la función implementada para predecir la categoría de los siguientes textos con ambos modelos (`LSTM` y `GRU`):

---

#### 📰 Ejemplo 1
- "The president met with international leaders to discuss global climate policies"

---

#### 🏅 Ejemplo 2
- "The team won the championship after a thrilling final match"

---

#### 💼 Ejemplo 3
- "The stock market saw a significant increase after the company reported strong earnings"

---

#### 🤖 Ejemplo 4
- "New advances in artificial intelligence are transforming the tech industry"

---

### 🔹 3. Comparación de resultados

👉 **Instrucción:**  
Para cada texto:

- Ejecuta la predicción con ambos modelos  
- Compara las clases predichas  
- Analiza si ambos modelos coinciden o si existen diferencias  

---

### ⚠️ Consideraciones importantes

- Debes utilizar el mismo `tokenizer` y `max_len` empleados durante el entrenamiento  
- Asegúrate de que el texto pase por el mismo proceso de preprocesamiento  
- Las predicciones deben interpretarse según el siguiente mapeo:

  - 0 → World  
  - 1 → Sports  
  - 2 → Business  
  - 3 → Sci/Tech  

---

### 🎯 Objetivo

Aplicar los modelos entrenados a datos nuevos y analizar su capacidad de generalización en escenarios reales.